# 1 · Semantic data-quality checks (silver layer)

Data-quality tools check types, nulls and ranges. They can't tell that `address` holds a phone number, that `product_description` says *zie mail*, or which code a free-text `cancel_reason` means.

**The pattern:** code finds the candidates and owns the workflow → Jev makes the semantic judgment → its probability decides *accept*, *reject* or *review*.

In [ ]:
import json

import pandas as pd

from jevdemo import dq
from jevdemo.judge import MODEL, Judge, LiveUnavailable, NotRecorded, request_body
from jevdemo.pipeline import bucket

judge = Judge()  # JEV_MODE=live for live calls; default replays recordings
print(f"mode: {judge.mode} · model: {MODEL}")

## The silver-layer rows
30 rows from a synthetic order table. The `is_*` and `cancel_code` columns are our hand labels, used only to score Jev afterwards.

In [ ]:
rows = dq.load_rows()
pd.DataFrame(rows)[["row_id", "address", "product_description", "cancel_reason"]]

## The checks are config, in plain language
Each check has a threshold on the fail rate and an action: `warn`, `quarantine` or `fail_run`.

In [ ]:
print(dq.CHECKS.read_text())
checks = dq.load_checks()

## What Jev receives
A yes/no check is one **Noul**: a probability that the value passes. Mapping to a code list is one **Choice**: a probability per code.

In [ ]:
for check in checks[0], checks[2]:
    value = rows[0][check.column]
    print(
        json.dumps(
            request_body(*dq.check_request(check, value)), indent=2, ensure_ascii=False
        )
    )

## Code step: ask once per distinct value, cache the answers
Answers are cached by `(check_id, value_hash)`, so only values never seen before cost a call.

In [ ]:
cache = dq.Cache()
judgments, stats = dq.judge_checks(rows, checks, judge, cache)
print(
    f"{len(rows)} rows × {len(checks)} checks = {len(rows) * len(checks)} cells → asked Jev {stats.asked} times"
)

## Tomorrow's run: same values, nothing new to ask

In [ ]:
_, rerun = dq.judge_checks(rows, checks, judge, cache)
print(f"asked {rerun.asked}, from cache {rerun.cached} → hit rate {rerun.hit_rate:.0%}")

## The threshold table
This is the part a pipeline owner decides. Change `LO` / `HI` and rerun this cell: no API calls.

In [ ]:
LO, HI = 0.10, 0.90

result = dq.run(rows, checks, judgments, lo=LO, hi=HI)
print(f"run status: {result.manifest['status'].upper()}")
pd.DataFrame(result.manifest["checks"]).T

## Quarantine table
Rows Jev rejected on a `quarantine` or `fail_run` check. They stay out of gold until fixed.

In [ ]:
pd.DataFrame(result.quarantine)[["row_id", "check_id", "value", "p_pass"]]

## Where Jev wasn't sure
These go to the review queue instead of being silently passed or quarantined.

In [ ]:
decided = pd.DataFrame(result.rows)
decided[decided["decision"] == "review"].drop_duplicates(["check_id", "value"])[
    ["check_id", "value", "p_pass", "code"]
]

## Free text → code
The Choice check gives a code for every cancel reason, and `other` when none fits.

In [ ]:
codes = decided[decided["check_id"] == "cancel_reason_code"].drop_duplicates("value")
codes[["value", "code", "p_pass", "decision"]]

## How did Jev do against our labels?
Only auto-decided rows count; review rows go to a person.

In [ ]:
labels = {r["row_id"]: r for r in rows}
truth = {
    "address_is_postal": lambda r: r["is_address"],
    "description_names_product": lambda r: r["is_product_description"],
    "cancel_reason_code": lambda r: r["cancel_code"] != "other",
}
auto = decided[decided["decision"] != "review"].copy()
auto["correct"] = [
    (d == "accept") == truth[c](labels[r])
    for c, d, r in zip(auto["check_id"], auto["decision"], auto["row_id"])
]
mapped = decided[decided["check_id"] == "cancel_reason_code"]
code_ok = sum(
    c == labels[r]["cancel_code"] for c, r in zip(mapped["code"], mapped["row_id"])
)
print(
    auto.groupby("check_id")["correct"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "correct", "count": "auto-decided"})
)
print(f"\ncancel codes matching the label: {code_ok} of {len(mapped)}")

## Your turn
Type an address. The preset is recorded; a new value needs `JEV_MODE=live`.

In [ ]:
value = dq.LIVE_PRESET  # or type your own

address = checks[0]
try:
    answer = judge.ask(*dq.check_request(address, value))
    p = dq.to_judgment(address, value, answer.response).p_pass
    print(
        f"p_pass={p:.2f} → {bucket(p, LO, HI)}  ({answer.source}, {answer.latency_ms} ms)"
    )
except (NotRecorded, LiveUnavailable) as e:
    print(e)